# M.M.M Make Minecraft Mode
### 역할 분리형 AI + 실제 MCP + Fabric 1.20.1 생성·빌드·검증

이 노트북은 저장소의 `config/model_registry.yaml`만 모델 설정 원본으로
사용합니다. 모델 로딩이나 생성에 실패하면 휴리스틱으로 몰래 전환하지
않고 즉시 실패 원인을 표시합니다.


In [ ]:
# @title 1. 제작 요청과 실행 설정
PROMPT = "서리 테마 아이템 2개와 블록 2개, 41x41 아레나가 있는 Fabric 1.20.1 모드를 만들어줘. 보스는 넣지 마." # @param {type:"string"}
MODEL_PROFILE = "t4_local" # @param ["t4_local", "remote_quality"]
REPO_REF = "main" # @param {type:"string"}
SOURCE_ONLY = False # @param {type:"boolean"}
APPROVE_PLAN = True # @param {type:"boolean"}
OUTPUT_ROOT = "/content/mmm-output" # @param {type:"string"}

if not PROMPT.strip():
    raise ValueError("PROMPT가 비어 있습니다.")
if not APPROVE_PLAN:
    print("계획만 생성합니다. 빌드하려면 APPROVE_PLAN=True로 명시하세요.")


In [ ]:
# @title 2. 저장소와 의존성 설치
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/M.M.M-Make-Mincraft-Mode")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    [
        "git", "clone", "--depth", "1", "--branch", REPO_REF,
        "https://github.com/jujumelona/M.M.M-Make-Mincraft-Mode.git",
        str(REPO_DIR),
    ],
    check=True,
)
os.chdir(REPO_DIR)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "-e",
        ".[ui,local-model,rag,image,speech]",
    ],
    check=True,
)

import torch
print("Python:", sys.version.split()[0])
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    free_bytes, total_bytes = torch.cuda.mem_get_info()
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM free/total: {free_bytes/2**30:.2f}/{total_bytes/2**30:.2f} GiB")


In [ ]:
# @title 3. 역할별 모델·MCP 설정 확인
import json
from minecraft_mod_ai import ModelRegistry

registry = ModelRegistry()
public_registry = registry.to_public_dict()
if MODEL_PROFILE not in public_registry["profiles"]:
    raise ValueError(
        f"알 수 없는 MODEL_PROFILE={MODEL_PROFILE!r}: "
        f"{sorted(public_registry['profiles'])}"
    )
print(json.dumps(
    public_registry["profiles"][MODEL_PROFILE],
    ensure_ascii=False,
    indent=2,
))
print("\nMCP 설정:")
print(Path(".mcp.json").read_text(encoding="utf-8"))


In [ ]:
# @title 4. 멀티모달 GameDesignPlanner 계획 생성
from minecraft_mod_ai import ModAISession

session = ModAISession.with_local_model(
    output_root=OUTPUT_ROOT,
    minecraft_version="1.20.1",
    profile=MODEL_PROFILE,
)
reply = session.plan(PROMPT)
print(reply.message)
print("\nready_to_build:", reply.ready_to_build)
if reply.questions:
    print("\n추가로 확정해야 할 내용:")
    for question in reply.questions:
        print("-", question)


In [ ]:
# @title 5. 승인된 계획으로 실제 생성·Gradle·GameTest 실행
BUILD_RESULT = None
if not APPROVE_PLAN:
    print("APPROVE_PLAN=False: 파일을 생성하지 않았습니다.")
elif not reply.ready_to_build:
    raise RuntimeError(
        "계획에 미확정 또는 아직 구현과 연결되지 않은 기능이 있습니다. "
        "PROMPT를 수정한 뒤 다시 실행하세요."
    )
else:
    BUILD_RESULT = session.build(
        reply,
        source_only=SOURCE_ONLY,
        output_root=OUTPUT_ROOT,
    )
    print(json.dumps(BUILD_RESULT.to_dict(), ensure_ascii=False, indent=2))
    if not SOURCE_ONLY and not BUILD_RESULT.release_ready:
        raise RuntimeError(
            "Gradle/GameTest/JAR 검증이 모두 통과하지 않아 설치용 릴리스를 "
            "완료하지 못했습니다. 생성된 로그와 report를 확인하세요."
        )


In [ ]:
# @title 6. 검증 결과 다운로드
if BUILD_RESULT is None:
    print("다운로드할 빌드 결과가 없습니다.")
else:
    release_zip = Path(BUILD_RESULT.release_zip)
    if not release_zip.is_file():
        raise FileNotFoundError(release_zip)
    print("release:", release_zip)
    print("size:", release_zip.stat().st_size, "bytes")
    try:
        from google.colab import files as colab_files
        colab_files.download(str(release_zip))
    except ImportError:
        print("로컬 경로:", release_zip.resolve())


## 통합 기능 실행 방법

기본 노트북은 승인된 Fabric 생성→정적 검증→Gradle→GameTest→JAR 검사 경로를
실행합니다. 코드 RAG, JDT, Blockbench, GeckoLib, WorldDesignIR→NBT/Jigsaw,
disposable runtime, Mineflayer, 검증 trace 수집은 `mmm-mcp` 도구로 분리되어
있으며 각 외부 실행 파일과 서버가 준비된 경우에만 성공합니다. 준비되지 않은
도구를 성공한 것으로 표시하지 않습니다.
